In [ ]:
import pandas as pd

PATH = "/content/skill_builder_data.csv"   # change to your file name

df = pd.read_csv(PATH, encoding="ISO-8859-15", usecols=["order_id", "user_id", "skill_id", "correct"])

print(df.shape)
print(df.head(10))
print(df.isna().sum())
print("students:", df["user_id"].nunique())
print("skills  :", df["skill_id"].nunique())

(303260, 4)
   order_id  user_id  correct  skill_id
0  33022537  64525.0      1.0       1.0
1  33022709  64525.0      1.0       1.0
2  35450204  70363.0      0.0       1.0
3  35450295  70363.0      1.0       1.0
4  35450311  70363.0      0.0       1.0
5  35450555  70363.0      1.0       1.0
6  35450573  70363.0      1.0       1.0
7  35480603  70363.0      1.0       1.0
8  33140811  70677.0      1.0       1.0
9  33140919  70677.0      1.0       1.0
order_id    0
user_id     1
correct     1
skill_id    1
dtype: int64
students: 3733
skills  : 82


In [ ]:
# 1. the broken row
print(df[df.isna().any(axis=1)])

# 2. what values does 'correct' have?
print(df["correct"].value_counts())

# 3. how long is each student's history?
print(df.groupby("user_id").size().describe())

# 4. duplicate rows?
print("duplicates:", df.duplicated(["user_id", "order_id", "skill_id"]).sum())

        order_id  user_id  correct  skill_id
303259      2719      NaN      NaN       NaN
correct
1.0    205040
0.0     98219
Name: count, dtype: int64
count    3733.000000
mean       81.237343
std       202.659193
min         1.000000
25%         8.000000
50%        22.000000
75%        75.000000
max      6149.000000
dtype: float64
duplicates: 46668


In [ ]:
import pandas as pd

df = pd.read_csv(PATH, encoding="ISO-8859-15", usecols=["order_id", "user_id", "skill_id", "correct"])
print("loaded       :", df.shape)

df = df.dropna()
df = df.drop_duplicates(["user_id", "order_id", "skill_id"])
df = df.astype({"user_id": int, "skill_id": int, "correct": int})
print("after clean  :", df.shape)

print("students     :", df["user_id"].nunique())
print("skills       :", df["skill_id"].nunique())
print("correct rate :", round(df["correct"].mean(), 3))

lens = df.groupby("user_id").size()
print("students with < 3 steps :", (lens < 3).sum())
print("students with > 200 steps:", (lens > 200).sum())

loaded       : (303260, 4)
after clean  : (256591, 4)
students     : 3733
skills       : 82
correct rate : 0.655
students with < 3 steps : 325
students with > 200 steps: 325


In [ ]:
skills = sorted(df["skill_id"].unique())
skill_map = {s: i for i, s in enumerate(skills)}
N = len(skills)

df = df.sort_values(["user_id", "order_id"], kind="stable")
df["sk"] = df["skill_id"].map(skill_map)
df["x"] = df["sk"] + N * df["correct"]

lens = df.groupby("user_id").size()
keep = lens[lens >= 3].index
df = df[df["user_id"].isin(keep)]
lens = lens[keep]

print("N =", N, "| max x =", df["x"].max())
print("students kept        :", len(keep))
print("rows total           :", len(df))
print("rows kept if cut 200 :", lens.clip(upper=200).sum())
print("chunks of 200 (B)    :", ((lens + 199) // 200).sum(), "sequences")

N = 82 | max x = 163
students kept        : 3408
rows total           : 256073
rows kept if cut 200 : 185659
chunks of 200 (B)    : 3957 sequences


In [ ]:
import numpy as np

MAX_LEN, MIN_LEN = 200, 3

# 1. split by STUDENT (so no student appears in both train and test)
users = np.array(sorted(keep))
rng = np.random.default_rng(42)
rng.shuffle(users)
cut = int(0.8 * len(users))
train_users, test_users = users[:cut], users[cut:]

# 2. one student -> one or more chunks of up to 200 steps, then pad
def build(user_list):
    g = {u: d for u, d in df.groupby("user_id")}
    chunks = []
    for u in user_list:
        d = g[u]
        for s in range(0, len(d), MAX_LEN):
            piece = d.iloc[s:s + MAX_LEN]
            if len(piece) >= MIN_LEN:
                chunks.append(piece)
    n = len(chunks)
    X, S, C, M = (np.zeros((n, MAX_LEN), dtype=np.int64) for _ in range(4))
    for i, p in enumerate(chunks):
        k = len(p)
        X[i, :k], S[i, :k], C[i, :k], M[i, :k] = p["x"], p["sk"], p["correct"], 1
    return X, S, C, M

X_tr, S_tr, C_tr, M_tr = build(train_users)
X_te, S_te, C_te, M_te = build(test_users)

# 3. checks
print("train students:", len(train_users), "| test students:", len(test_users))
print("train shape   :", X_tr.shape, "| test shape:", X_te.shape)
print("real steps    : train", M_tr.sum(), "| test", M_te.sum(), "| total", M_tr.sum() + M_te.sum())
print("x formula ok  :", bool(((S_tr + N * C_tr)[M_tr == 1] == X_tr[M_tr == 1]).all()))
print("correct rate  : train", round(C_tr[M_tr == 1].mean(), 3), "| test", round(C_te[M_te == 1].mean(), 3))

np.savez("dkt_data.npz", X_tr=X_tr, S_tr=S_tr, C_tr=C_tr, M_tr=M_tr,
         X_te=X_te, S_te=S_te, C_te=C_te, M_te=M_te, N=N)

train students: 2726 | test students: 682
train shape   : (3168, 200) | test shape: (781, 200)
real steps    : train 205418 | test 50642 | total 256060
x formula ok  : True
correct rate  : train 0.657 | test 0.644


In [ ]:
import os, hashlib, shutil, numpy as np, pandas as pd
from google.colab import files

OUT = "/content/processed"
os.makedirs(OUT, exist_ok=True)

# ----- which student does each chunk belong to -----
def chunk_users(user_list):
    g = {u: d for u, d in df.groupby("user_id")}
    out = []
    for u in user_list:
        d = g[u]
        for s in range(0, len(d), MAX_LEN):
            if len(d.iloc[s:s + MAX_LEN]) >= MIN_LEN:
                out.append(u)
    return out

def to_table(split, X, S, C, M, users_of_chunk):
    assert len(users_of_chunk) == X.shape[0], "chunk count mismatch"
    seq, step = np.nonzero(M)                     # only real steps, no padding
    return pd.DataFrame({
        "split": split,
        "user_id": np.array(users_of_chunk)[seq],
        "chunk_id": seq,
        "step": step,
        "skill_index": S[seq, step],
        "correct": C[seq, step],
        "x": X[seq, step],
    })

tab = pd.concat([
    to_table("train", X_tr, S_tr, C_tr, M_tr, chunk_users(train_users)),
    to_table("test",  X_te, S_te, C_te, M_te, chunk_users(test_users)),
], ignore_index=True)

inv = {i: s for s, i in skill_map.items()}
tab["skill_id_original"] = tab["skill_index"].map(inv)
tab.to_csv(f"{OUT}/dkt_processed_steps.csv", index=False)

pd.DataFrame({"skill_index": list(inv), "skill_id_original": list(inv.values())}
             ).to_csv(f"{OUT}/skill_map.csv", index=False)

np.savez_compressed(f"{OUT}/dkt_data.npz", X_tr=X_tr, S_tr=S_tr, C_tr=C_tr, M_tr=M_tr,
                    X_te=X_te, S_te=S_te, C_te=C_te, M_te=M_te, N=N)

# ----- fingerprint, to prove the data is not changed after submission -----
with open(f"{OUT}/SHA256.txt", "w") as f:
    for name in ("dkt_processed_steps.csv", "skill_map.csv", "dkt_data.npz"):
        h = hashlib.sha256(open(f"{OUT}/{name}", "rb").read()).hexdigest()
        f.write(f"{h}  {name}\n")

# ----- reload and verify -----
z = np.load(f"{OUT}/dkt_data.npz")
ok = all((z[k] == v).all() for k, v in dict(X_tr=X_tr, S_tr=S_tr, C_tr=C_tr, M_tr=M_tr,
                                              X_te=X_te, S_te=S_te, C_te=C_te, M_te=M_te).items())
print("rows in csv    :", len(tab), "| train", (tab.split == "train").sum(), "| test", (tab.split == "test").sum())
print("students in csv:", tab.user_id.nunique())
print("x formula ok   :", bool((tab.skill_index + N * tab.correct == tab.x).all()))
print("npz reload ok  :", ok)
print(open(f"{OUT}/SHA256.txt").read())
print(os.listdir(OUT))

# ----- zip and download to your laptop -----
shutil.make_archive("/content/dkt_processed", "zip", OUT)
files.download("/content/dkt_processed.zip")

rows in csv    : 256060 | train 205418 | test 50642
students in csv: 3408
x formula ok   : True
npz reload ok  : True
a804d21e9afa419134b0543ea997bdfe0774e169d3114d700b540fece34af2ee  dkt_processed_steps.csv
9fd7c70a10d4626fe185a56c0b194697e9fc4f90288abbb9332bf79b931f8862  skill_map.csv
a261bd561c671209420773f34015e2937f20e88ecb7657fc81cb03f8c7196d12  dkt_data.npz

['dkt_processed_steps.csv', 'skill_map.csv', 'SHA256.txt', 'dkt_data.npz']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !pip install -q datasets

# from datasets import load_dataset

# dataset = load_dataset("ulises-c/SocraTeach_Multi")

# print(dataset)
# print(dataset["train"].column_names)

# # Inspect one example
# sample = dataset["train"][0]

# print("Question:", sample["question"])
# print("Available fields:", sample.keys())
# print("Dialogue example:", sample["dialogues"][0])

README.md:   0%|          | 0.00/4.27k [00:00<?, ?B/s]

SocraTeach_multi.jsonl: reconstructing file:   0%|          |  0.00B / 58.8MB            

SocraTeach_multi.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10273 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'analysis', 'answer', 'steps', 'dialogues'],
        num_rows: 10273
    })
})
['id', 'question', 'analysis', 'answer', 'steps', 'dialogues']
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Available fields: dict_keys(['id', 'question', 'analysis', 'answer', 'steps', 'dialogues'])
Dialogue example: {'dialogue_id': 'GSM8K_train_0_0', 'turns': [{'system': 'How many clips did Natalia sell in May?', 'user': 'I\'m confused. Does "half as many" mean the same number as April?', 'user_type': '(1)'}, {'system': 'Not exactly, "half as many" means half the amount. If Natalia sold 48 clips in April, what would be half of 48? That would tell us how many she sold in May. Can you solve that?', 'user': " I'm still not sure. Is it like 48 divided by 2?", 'user_type': '(4)'}, {'system': "Exactly, that's right! If we 

In [ ]:

from datasets import load_dataset

DATASET_NAME = "ulises-c/SocraTeach_Multi"

dataset = load_dataset(DATASET_NAME)

print(dataset)
print("\nAvailable splits:", list(dataset.keys()))


README.md:   0%|          | 0.00/4.27k [00:00<?, ?B/s]

SocraTeach_multi.jsonl: reconstructing file:   0%|          |  0.00B / 58.8MB            

SocraTeach_multi.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10273 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'analysis', 'answer', 'steps', 'dialogues'],
        num_rows: 10273
    })
})

Available splits: ['train']


In [ ]:

sample = dataset["train"][0]

print("Example ID:", sample["id"])
print("\nQUESTION:")
print(sample["question"])

print("\nANALYSIS:")
print(str(sample["analysis"])[:1000])

print("\nANSWER:")
print(sample["answer"])

print("\nSTEPS:")
print(str(sample["steps"])[:1000])

print("\nNUMBER OF DIALOGUES:", len(sample["dialogues"]))

print("\nFIRST DIALOGUE:")
print(sample["dialogues"][0])


Example ID: GSM8K_train_0

QUESTION:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

ANALYSIS:
 Natalia sold 48/2 = 24 clips in May. Natalia sold 48+24 = 72 clips altogether in April and May.

ANSWER:
72

STEPS:
['How many clips did Natalia sell in May?', 'How many clips did Natalia sell altogether in April and May?']

NUMBER OF DIALOGUES: 5

FIRST DIALOGUE:
{'dialogue_id': 'GSM8K_train_0_0', 'turns': [{'system': 'How many clips did Natalia sell in May?', 'user': 'I\'m confused. Does "half as many" mean the same number as April?', 'user_type': '(1)'}, {'system': 'Not exactly, "half as many" means half the amount. If Natalia sold 48 clips in April, what would be half of 48? That would tell us how many she sold in May. Can you solve that?', 'user': " I'm still not sure. Is it like 48 divided by 2?", 'user_type': '(4)'}, {'system': "Exactly, that's right! If we divide 48 by 2, we g

In [ ]:

from collections import Counter

train_data = dataset["train"]

dialogue_counts = []
turn_counts = []
end_turn_counts = Counter()
empty_turns = 0

for example in train_data:
    dialogues = example.get("dialogues") or []
    dialogue_counts.append(len(dialogues))

    for dialogue in dialogues:
        turns = dialogue.get("turns") or []
        turn_counts.append(len(turns))

        for turn in turns:
            if not turn.get("system"):
                empty_turns += 1

            if turn.get("END") == "True":
                end_turn_counts["END=True"] += 1

print("Total examples:", len(train_data))
print("Total dialogues:", sum(dialogue_counts))
print("Total individual turns:", sum(turn_counts))
print("Average dialogues per example:", round(sum(dialogue_counts) / len(train_data), 2))
print("Average turns per dialogue:", round(sum(turn_counts) / len(turn_counts), 2))
print("Turns missing tutor/system text:", empty_turns)
print("Final END turns:", end_turn_counts["END=True"])
print("Dialogue count distribution:", Counter(dialogue_counts))


Total examples: 10273
Total dialogues: 35151
Total individual turns: 185602
Average dialogues per example: 3.42
Average turns per dialogue: 5.28
Turns missing tutor/system text: 2
Final END turns: 35151
Dialogue count distribution: Counter({5: 2712, 4: 2600, 3: 2139, 2: 1532, 1: 1206, 6: 84})


In [ ]:

# Inspect the final turn from three dialogues
count = 0

for example in dataset["train"]:
    for dialogue in example.get("dialogues", []):
        turns = dialogue.get("turns", [])

        if turns and turns[-1].get("END") == "True":
            print("Dialogue ID:", dialogue.get("dialogue_id"))
            print("Final turn:")
            print(turns[-1])
            print("-" * 70)

            count += 1

            if count == 3:
                break

    if count == 3:
        break


Dialogue ID: GSM8K_train_0_0
Final turn:
{'END': 'True', 'system': "That's right! Good job! So, if Natalia sold 48 clips in April and 24 clips in May, she did indeed sell 72 clips in total. Well done!"}
----------------------------------------------------------------------
Dialogue ID: GSM8K_train_0_1
Final turn:
{'END': 'True', 'system': "Well done! You added the number of clips sold in April and May correctly. You've understood it perfectly. Is there anything else you want to know about this topic or any other topic?"}
----------------------------------------------------------------------
Dialogue ID: GSM8K_train_0_2
Final turn:
{'END': 'True', 'system': "Perfect! You've arrived at the correct answer. By first dividing the number of clips Natalia sold in April by 2 and then adding that number to the clips sold in April, you have found out the total number of clips sold across both months. So, do you now understand the process of using these math steps to solve real-life problems?"}
-

In [ ]:

import json
from collections import Counter

processed_data = []
stats = Counter()

for example in dataset["train"]:
    for dialogue in (example.get("dialogues") or []):
        turns = dialogue.get("turns") or []
        messages = []

        for turn in turns:
            tutor_text = turn.get("system")
            student_text = turn.get("user")

            # Keep the tutor's message
            if isinstance(tutor_text, str) and tutor_text.strip():
                messages.append({
                    "role": "assistant",
                    "content": tutor_text.strip()
                })
            else:
                stats["missing_tutor_messages"] += 1

            # Keep a student reply only when one exists
            if isinstance(student_text, str) and student_text.strip():
                messages.append({
                    "role": "user",
                    "content": student_text.strip()
                })

        # Keep only dialogues containing at least one user and tutor message
        roles = {message["role"] for message in messages}

        if len(messages) >= 2 and roles == {"user", "assistant"}:
            processed_data.append({
                "dialogue_id": dialogue.get("dialogue_id"),
                "messages": messages
            })
        else:
            stats["skipped_dialogues"] += 1

print("Processed dialogues:", len(processed_data))
print("Missing tutor messages:", stats["missing_tutor_messages"])
print("Skipped dialogues:", stats["skipped_dialogues"])

print("\nFirst processed record:")
print(json.dumps(processed_data[0], indent=2, ensure_ascii=False)[:3000])


Processed dialogues: 35151
Missing tutor messages: 2
Skipped dialogues: 0

First processed record:
{
  "dialogue_id": "GSM8K_train_0_0",
  "messages": [
    {
      "role": "assistant",
      "content": "How many clips did Natalia sell in May?"
    },
    {
      "role": "user",
      "content": "I'm confused. Does \"half as many\" mean the same number as April?"
    },
    {
      "role": "assistant",
      "content": "Not exactly, \"half as many\" means half the amount. If Natalia sold 48 clips in April, what would be half of 48? That would tell us how many she sold in May. Can you solve that?"
    },
    {
      "role": "user",
      "content": "I'm still not sure. Is it like 48 divided by 2?"
    },
    {
      "role": "assistant",
      "content": "Exactly, that's right! If we divide 48 by 2, we get the number of clips Natalia sold in May. What number do you get when you do that?"
    },
    {
      "role": "user",
      "content": "Oh, I get it now. So 48 divided by 2 is 28."
   

In [ ]:

import json
from collections import Counter

role_counts = Counter()
empty_messages = 0
invalid_records = 0
duplicate_ids = 0

seen_ids = set()

for record in processed_data:
    dialogue_id = record.get("dialogue_id")
    messages = record.get("messages", [])

    if not dialogue_id or not messages:
        invalid_records += 1
        continue

    if dialogue_id in seen_ids:
        duplicate_ids += 1
    seen_ids.add(dialogue_id)

    for message in messages:
        role_counts[message.get("role")] += 1

        if not message.get("content", "").strip():
            empty_messages += 1

print("Total processed dialogues:", len(processed_data))
print("Total messages:", sum(role_counts.values()))
print("Role distribution:", dict(role_counts))
print("Empty messages:", empty_messages)
print("Invalid records:", invalid_records)
print("Duplicate dialogue IDs:", duplicate_ids)

# Check that every record can be serialized as JSON
try:
    for record in processed_data:
        json.dumps(record, ensure_ascii=False)
    print("JSON serialization: PASSED")
except Exception as e:
    print("JSON serialization: FAILED")
    print(type(e).__name__, str(e))


Total processed dialogues: 35151
Total messages: 336051
Role distribution: {'assistant': 185600, 'user': 150451}
Empty messages: 0
Invalid records: 0
Duplicate dialogue IDs: 0
JSON serialization: PASSED


In [ ]:

import json
from google.colab import files

output_file = "soctrateach_processed.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for record in processed_data:
        # Qwen chat format: one conversation per JSONL line
        qwen_record = {"messages": record["messages"]}
        f.write(json.dumps(qwen_record, ensure_ascii=False) + "\n")

print("File created:", output_file)

# Verify the exported file
with open(output_file, "r", encoding="utf-8") as f:
    first_line = f.readline()
    total_lines = 1 + sum(1 for _ in f)

first_record = json.loads(first_line)

print("Total exported conversations:", total_lines)
print("First record keys:", list(first_record.keys()))
print("First conversation messages:", len(first_record["messages"]))
print("File format check:", "PASSED" if total_lines == len(processed_data) else "FAILED")

# Download to your computer
files.download(output_file)


File created: soctrateach_processed.jsonl
Total exported conversations: 35151
First record keys: ['messages']
First conversation messages: 11
File format check: PASSED


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

!pip -q install -U datasets

from datasets import load_dataset

MATHDIAL_NAME = "eth-nlped/mathdial-chat"

mathdial = load_dataset(MATHDIAL_NAME)

print("Dataset loaded successfully!")
print(mathdial)

print("\nAvailable splits:", list(mathdial.keys()))

for split_name, split_data in mathdial.items():
    print(f"\n{split_name}: {len(split_data)} rows")
    print("Columns:", split_data.column_names)

# Inspect one training example
sample = mathdial["train"][0]

print("\nFirst training example:")
for key, value in sample.items():
    print(f"\n{key}:")
    print(str(value)[:1800])


README.md:   0%|          | 0.00/759 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.64MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  650kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/599 [00:00<?, ? examples/s]

Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['question', 'ground_truth', 'conversation', 'student_incorrect_solution', 'student_profile'],
        num_rows: 2262
    })
    test: Dataset({
        features: ['question', 'ground_truth', 'conversation', 'student_incorrect_solution', 'student_profile'],
        num_rows: 599
    })
})

Available splits: ['train', 'test']

train: 2262 rows
Columns: ['question', 'ground_truth', 'conversation', 'student_incorrect_solution', 'student_profile']

test: 599 rows
Columns: ['question', 'ground_truth', 'conversation', 'student_incorrect_solution', 'student_profile']

First training example:

question:
Nancy is filling an aquarium for her fish. She fills it halfway and goes to answer the door. While she's gone, her cat knocks the aquarium over and spills half the water in it. Then Nancy comes back and triples the amount of water in the aquarium. If the aquarium is 4 feet long, 6 feet wide, and 3 feet high, how m

In [ ]:

from collections import Counter

train = mathdial["train"]

role_counts = Counter()
empty_messages = 0
invalid_conversations = 0
conversation_lengths = []

for row in train:
    conversation = row.get("conversation")

    if not isinstance(conversation, list) or not conversation:
        invalid_conversations += 1
        continue

    conversation_lengths.append(len(conversation))

    for message in conversation:
        role_counts[message.get("role", "MISSING")] += 1

        if not str(message.get("content", "")).strip():
            empty_messages += 1

print("Training examples:", len(train))
print("Role distribution:", dict(role_counts))
print("Empty messages:", empty_messages)
print("Invalid conversations:", invalid_conversations)

print(
    "Average messages per conversation:",
    round(sum(conversation_lengths) / len(conversation_lengths), 2)
    if conversation_lengths else 0
)

print("\nFirst conversation roles:")
print([message.get("role") for message in train[0]["conversation"]])


Training examples: 2262
Role distribution: {'system': 2262, 'assistant': 14452, 'user': 13410}
Empty messages: 4
Invalid conversations: 0
Average messages per conversation: 13.32

First conversation roles:
['system', 'assistant', 'user', 'assistant', 'user', 'assistant']


In [ ]:

import json
from collections import Counter
from google.colab import files

mathdial_processed = []
stats = Counter()

for index, row in enumerate(mathdial["train"]):
    conversation = row.get("conversation") or []
    messages = []

    for message in conversation:
        role = message.get("role")
        content = message.get("content")

        # Keep only valid Qwen chat roles
        if role not in {"system", "user", "assistant"}:
            stats["invalid_role_messages"] += 1
            continue

        # Remove empty or whitespace-only messages
        if not isinstance(content, str) or not content.strip():
            stats["empty_messages_removed"] += 1
            continue

        messages.append({
            "role": role,
            "content": content.strip()
        })

    # Keep conversations with a user and assistant message
    roles = {m["role"] for m in messages}

    if len(messages) >= 2 and "user" in roles and "assistant" in roles:
        mathdial_processed.append({"messages": messages})
    else:
        stats["conversations_skipped"] += 1

output_file = "mathdial_processed.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for record in mathdial_processed:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Validate exported JSONL
with open(output_file, "r", encoding="utf-8") as f:
    exported_lines = f.readlines()

assert len(exported_lines) == len(mathdial_processed)

for line in exported_lines:
    record = json.loads(line)
    assert isinstance(record["messages"], list)
    assert all(
        m["role"] in {"system", "user", "assistant"}
        and m["content"].strip()
        for m in record["messages"]
    )

print("Original training examples:", len(mathdial["train"]))
print("Processed conversations:", len(mathdial_processed))
print("Empty messages removed:", stats["empty_messages_removed"])
print("Invalid-role messages removed:", stats["invalid_role_messages"])
print("Conversations skipped:", stats["conversations_skipped"])
print("Exported JSONL lines:", len(exported_lines))
print("Validation: PASSED")
print("\nFirst processed conversation:")
print(json.dumps(mathdial_processed[0], indent=2, ensure_ascii=False)[:2000])

files.download(output_file)


Original training examples: 2262
Processed conversations: 2262
Empty messages removed: 4
Invalid-role messages removed: 0
Conversations skipped: 0
Exported JSONL lines: 2262
Validation: PASSED

First processed conversation:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a friendly and supportive teacher.\nThe student is trying to solve the following problem: Nancy is filling an aquarium for her fish. She fills it halfway and goes to answer the door. While she's gone, her cat knocks the aquarium over and spills half the water in it. Then Nancy comes back and triples the amount of water in the aquarium. If the aquarium is 4 feet long, 6 feet wide, and 3 feet high, how many cubic feet of water are in the aquarium?."
    },
    {
      "role": "assistant",
      "content": "Steven, If you had 4 of something and tripled that amount, how much would you have?"
    },
    {
      "role": "user",
      "content": "I would have 12 of something."
    },
    {
      "rol

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import json
from collections import Counter
from google.colab import files

combined_data = []
stats = Counter()

# Add SocraTeach dialogues
for record in processed_data:
    messages = record["messages"]

    # Validate required chat roles
    if not messages or not any(m["role"] == "assistant" for m in messages):
        stats["socra_invalid"] += 1
        continue

    combined_data.append({
        "source": "soctrateach",
        "messages": messages
    })

# Add MathDial conversations
for record in mathdial_processed:
    messages = record["messages"]

    if not messages or not any(m["role"] == "assistant" for m in messages):
        stats["mathdial_invalid"] += 1
        continue

    combined_data.append({
        "source": "mathdial",
        "messages": messages
    })

# Export combined dataset
output_file = "socratic_tutor_combined.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for record in combined_data:
        # Save source for tracking; training pipeline can remove it later
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Verify file integrity
with open(output_file, "r", encoding="utf-8") as f:
    exported = [json.loads(line) for line in f if line.strip()]

source_counts = Counter(record["source"] for record in exported)

print("SocraTeach processed:", len(processed_data))
print("MathDial processed:", len(mathdial_processed))
print("Combined conversations:", len(exported))
print("Source counts:", dict(source_counts))
print("Invalid SocraTeach records skipped:", stats["socra_invalid"])
print("Invalid MathDial records skipped:", stats["mathdial_invalid"])
print("All records valid:", all(
    isinstance(r.get("messages"), list)
    and all(
        m.get("role") in {"system", "user", "assistant"}
        and isinstance(m.get("content"), str)
        and m["content"].strip()
        for m in r["messages"]
    )
    for r in exported
))

print("\nFirst combined record source:", exported[0]["source"])
print("First combined record message count:", len(exported[0]["messages"]))

files.download(output_file)


SocraTeach processed: 35151
MathDial processed: 2262
Combined conversations: 37413
Source counts: {'soctrateach': 35151, 'mathdial': 2262}
Invalid SocraTeach records skipped: 0
Invalid MathDial records skipped: 0
All records valid: True

First combined record source: soctrateach
First combined record message count: 11


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>